<a href="https://colab.research.google.com/github/MohHaroon/XAI-based-ZSL-for-IIDS/blob/master/ESZSL_ZDBERTa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import pandas as pd
SDN_train_test_data = pd.read_csv("/content/drive/MyDrive/IRP/SDN-Dataset/SDN_data.csv")

In [11]:
print(SDN_train_test_data.columns)

Index(['proto_number', 'Dur', 'Mean', 'Stddev', 'Min', 'Max', 'Pkts', 'Bytes',
       'Spkts', 'Dpkts', 'Sbytes', 'Dbytes', 'Srate', 'Drate', 'Sum',
       'TnBPSrcIP', 'TnBPDstIP', 'TnP_PSrcIP', 'TnP_PDstIP', 'TnP_PerProto',
       'TnP_Per_Dport', 'N_IN_Conn_P_DstIP', 'N_IN_Conn_P_SrcIP', 'Attack'],
      dtype='object')


In [ ]:
display(SDN_train_test_data)

In [6]:
def scale_data(data):
    scaler = MinMaxScaler()
    scaled_data = scaler.fit_transform(data)
    return scaled_data

SDN_scaled_data = scale_data(SDN_train_test_data.drop(columns=['Attack']))

def split_data(data, labels, test_size=0.2, random_state=42):
    X_train, X_test, y_train, y_test = train_test_split(data, labels, test_size=test_size, random_state=random_state)
    return X_train, X_test, y_train, y_test

data_train, data_test, labels_train, labels_test = split_data(SDN_scaled_data, SDN_train_test_data['Attack'])

## Run system

#### ESZSL

In [24]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelBinarizer

# Updated Mapping for all categories
id_to_name = {
    0.0: "Normal",
    1.0: "DoS",
    2.0: "DDoS",
    3.0: "Port Scan",
    4.0: "Fuzzing",
    5.0: "OS Fingerprinting (Zero-Day)"
}

# Update S_matrix with all 6 rows
attributes_all = np.array([
    [0, 0, 0, 0, 1], # 0.0
    [1, 1, 0, 0, 0], # 1.0
    [1, 1, 0, 0, 0], # 2.0
    [0, 1, 1, 0, 0], # 3.0
    [0, 0, 0, 1, 0], # 4.0
    [0, 1, 1, 0, 1]  # 5.0
])

S_matrix = pd.DataFrame(attributes_all, index=[0.0, 1.0, 2.0, 3.0, 4.0, 5.0])

# 2. Get the unique labels present in your training split
seen_classes = np.unique(labels_train)
print(f"Seen classes in training: {seen_classes}")

# 3. Slice S_seen safely
S_seen = S_matrix.loc[seen_classes].values

# 4. Initialize and Train
model_eszsl = ESZSL_IOT(alpha=2, gamma=2)
model_eszsl.fit(data_train, labels_train, S_seen)

Seen classes in training: [0. 1.]


In [25]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder

class ESZSL_IOT:
    def __init__(self, alpha=2, gamma=2):
        self.alpha = alpha  # Regularization for feature space
        self.gamma = gamma  # Regularization for attribute space
        self.W = None

    def fit(self, X, y, S_seen):
        """
        X: Training features (N, D)
        y: Training labels (N, 1)
        S_seen: Attribute matrix for seen classes (L_seen, A)
        """
        # Convert y to one-hot binary matrix Y (N, L_seen)
        ohe = OneHotEncoder(sparse_output=False)
        Y = ohe.fit_transform(y.values.reshape(-1, 1))

        d = X.shape[1]
        a = S_seen.shape[1]

        # Solving the Sylvester Equation for W: X.T @ X @ W @ S @ S.T + ...
        # Simplified closed-form solution:
        part_1 = np.linalg.pinv(X.T @ X + (10**self.alpha) * np.eye(d))
        part_0 = X.T @ Y @ S_seen
        part_2 = np.linalg.pinv(S_seen.T @ S_seen + (10**self.gamma) * np.eye(a))

        self.W = part_1 @ part_0 @ part_2

    def predict(self, X, S_all):
        """Returns the index of the most similar class in S_all."""
        # Project into attribute space: Scores = X * W * S_all.T
        scores = X @ self.W @ S_all.T
        return np.argmax(scores, axis=1)

# # Usage Setup
# # attributes_all should be a (Classes x Attributes) matrix you defined previously
# attributes_all = np.array([
#     [0, 0, 0, 0, 1], # 0.0
#     [1, 1, 0, 0, 0], # 1.0
#     [1, 1, 0, 0, 0], # 2.0
#     [0, 1, 1, 0, 0], # 3.0
#     [0, 0, 0, 1, 0], # 4.0
#     [0, 1, 1, 0, 1]  # 5.0
# ])
S_matrix = pd.DataFrame(attributes_all, index=[0.0, 1.0, 2.0, 3.0, 4.0, 5.0])

#### ZDBERTa

In [13]:
from transformers import pipeline
import torch

class ZDBERTa_Predictor:
    def __init__(self, model_name="facebook/bart-large-mnli"):
        # Load NLI-based zero-shot classification pipeline
        self.device = 0 if torch.cuda.is_available() else -1
        self.classifier = pipeline("zero-shot-classification", model=model_name, device=self.device)

    def serialize(self, row):
        """Converts tabular SDN features into a natural language description."""
        return (f"Network flow protocol {int(row['proto_number'])} lasted {row['Dur']:.4f}s. "
                f"Volume: {int(row['Pkts'])} packets, {int(row['Bytes'])} bytes. "
                f"Source rate: {row['Srate']:.2f} pkts/sec. "
                f"Inbound connections: {int(row['N_IN_Conn_P_SrcIP'])}.")

    def predict_batch(self, X_df, candidate_labels):
        """Predicts labels for a batch of rows."""
        sentences = X_df.apply(self.serialize, axis=1).tolist()
        results = self.classifier(sentences, candidate_labels, multi_label=False)
        return [res['labels'][0] for res in results]

# Usage Setup
zdberta = ZDBERTa_Predictor()
labels = ["Normal", "DoS Attack", "DDoS", "Port Scan","Fuzzing", "OS Fingerprinting"]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

#### Test

In [26]:
import time

def run_benchmarks(X_test, y_test, eszsl_model, zdberta_model, S_matrix, labels_text):
    results = []

    # 1. Benchmark ESZSL (Fast Linear)
    start_eszsl = time.perf_counter()
    eszsl_preds = eszsl_model.predict(X_test, S_matrix.values)
    end_eszsl = time.perf_counter()

    # 2. Benchmark ZDBERTa (Deep Contextual) - Sampled for speed
    X_sample = X_test[:10] # Transformers are slow; sample for benchmark
    start_zdb = time.perf_counter()
    zdb_preds = zdberta_model.predict_batch(X_sample, labels_text)
    end_zdb = time.perf_counter()

    avg_lat_eszsl = ((end_eszsl - start_eszsl) / len(X_test)) * 1000
    avg_lat_zdb = ((end_zdb - start_zdb) / len(X_sample)) * 1000

    print(f"ESZSL Latency: {avg_lat_eszsl:.4f} ms/packet")
    print(f"ZDBERTa Latency: {avg_lat_zdb:.4f} ms/packet")

    return {"eszsl_lat": avg_lat_eszsl, "zdb_lat": avg_lat_zdb}

In [27]:
import pandas as pd

# Get the original column names from SDN_train_test_data, excluding the 'Attack' column
original_columns = SDN_train_test_data.drop(columns=['Attack']).columns

# Convert data_test (which is a NumPy array) into a Pandas DataFrame
# This ensures that the .apply() method and column-name-based access in serialize() will work.
data_test_df = pd.DataFrame(data_test, columns=original_columns)

# Now, call run_benchmarks with the DataFrame version of data_test
execution_results = run_benchmarks(data_test_df, labels_test, model_eszsl, zdberta, S_matrix, labels)

ESZSL Latency: 0.0003 ms/packet
ZDBERTa Latency: 5176.1035 ms/packet


In [ ]:
import pandas as pd

# Convert the dictionary to a pandas DataFrame
results_df = pd.DataFrame([execution_results])

# Now, save the DataFrame to CSV
results_df.to_csv('ESZSL_ZDBERTa_Results.csv', index=False)
print("Comprehensive Stress Test Complete.")